# RCIS Scheduler
- Uses a FCFS (First Come, First Serve) Algorithm to determine interviewee bookings for companies

## Algo Approach
- For each day, get companies and interviewees available
- For each company in that day, get all timeslots where company is available.
- Filter interviewee list (form responses) for each timeslot + company preferred degree program, order based on response time
- Get interviewee at the top, remove interviewee from list


## Dependency Installation

In [1]:
%pip install pandas

  Using cached pandas-2.3.3-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached numpy-2.3.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached pytz-2025.2-py2.py3-none-any.whl.metadata (22 kB)
Using cached pandas-2.3.3-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.3 MB)
Using cached numpy-2.3.4-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.6 MB)
Using cached pytz-2025.2-py2.py3-none-any.whl (509 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pandas]━━━━━━━━━━━ 2/3 [pandas]
Note: you may need to restart the kernel to use updated packages.


## Imports

In [ ]:
from datetime import datetime
from typing import NewType, Protocol
from enum import Enum

## Classes and Protocols for Scheduler

In [ ]:
class DegreeProgram(str, Enum):
    ChemicalEngg = 'BS Chemical Engineering'
    CivilEngg = 'BS Civil Engineering'
    ComputerEngg = 'BS Computer Engineering'
    ComputerScience = 'BS Computer Science'
    ElectricalEngg = 'BS Electrical Engineering'
    ElectronicsEngg = 'BS Electronics Engineering'
    GeodeticEngg = 'BS Geodetic Engineering'
    IndustrialEngg = 'BS Industrial Engineering'
    MaterialsEngg = 'BS Materials Engineering'
    MechEngg = 'BS Mechanical Engineering'
    MetalEngg = 'BS Metallurgical Engineering'
    MiningEngg = 'BS Mining Engineering'

class Times(str, Enum): # not ideal, but I don't really want to deal with datetime right now
# Resume Consultations
    Time0915_0945 = "09:00 AM - 09:45 AM"
    Time1000_1045 = "10:00 AM - 10:45 AM"
    Time1100_1145 = "11:00 AM - 11:45 AM"
    Time1315_1400 = "01:15 PM - 02:00 PM"
    Time1415_1500 = "02:15 PM - 03:00 PM"
    Time1515_1600 = "03:15 PM - 04:00 PM"
    Time1615_1700 = "04:15 PM - 05:00 PM"

# Interview Simulations
    Time0900_0930 = "09:00 AM - 09:30 AM"
    Time0930_1000 = "09:30 AM - 10:00 AM"
    Time1015_1045 = "10:15 AM - 10:45 AM"
    Time1045_1115 = "10:45 AM - 11:15 AM"
    Time1130_1200 = "11:30 AM - 12:00 PM"
    Time1330_1400 = "01:30 PM - 02:00 PM"
    Time1400_1430 = "02:00 PM - 02:30 PM"
    Time1445_1515 = "02:45 PM - 03:15 PM"
    Time1515_1545 = "03:15 PM - 03:45 PM"
    Time1600_1630 = "04:00 PM - 04:30 PM"
    Time1445_1515 = "04:30 PM - 05:00 PM"
class Participant(Protocol):
    @property
    def name(self) -> str:
        # Return str that is the name of the participant (could be interviewee/interviewer)
        ...
    @property
    def time_slots(self) -> list[Times]:
        # Returns time slots of participant
        ...
class Interviewee:
    def __init__(self, name: str, email: str, capes_id: str, time_slots: list[Times], registration_order: int, degree_program: DegreeProgram):
        self._name = name
        self._email = email
        self._capes_id = capes_id
        self._time_slots = time_slots
        self._degree_program = degree_program
        self._priority = len(time_slots)
        self._registration_order = registration_order
        self._granted_slot: Times
    def __eq__(self, b):
        return type(self) == type(b) and self.name == b.name 
    def __hash__(self):
        return hash(self.name)
    @property
    def name(self) -> str:
        return self._name
    @property
    def email(self) -> str:
        return self._email
    @property
    def capes_id(self) -> str:
        return self._capes_id
    @property
    def time_slots(self) -> list[Times]:
        return self._time_slots
    @property
    def degree_program(self) -> DegreeProgram:
        return self._degree_program
    @property
    def priority(self) -> int:
        return self._priority
    @property
    def registration_order(self) -> int:
        return self._registration_order
    @property
    def granted_slot(self) -> Times:
        return self._granted_slot
    def set_granted_slot(self, time: Times):
        self._granted_slot = time
class Interviewer:
    def __init__(self, name: str, time_slots: list[Times], degree_program_preference: list[DegreeProgram] | DegreeProgram, appointment_type : str):
        self._name = name
        self._time_slots = time_slots
        self._degree_program_preference = degree_program_preference
        self._interviewee_list : dict[Times, list[Interviewee]] = dict()
        self._appointment_type = appointment_type
    def __lt__(a : Interviewer, b : Interviewer):
        return a.priority < b.priority
    def unallocated_type(self, type):
        self._unallocated_type = type
    @property
    def name(self) -> str:
        return self._name
    @property
    def time_slots(self) -> list[Times]:
        return self._time_slots
    @property
    def degree_program_preference(self) -> list[DegreeProgram] | DegreeProgram:
        return self._degree_program_preference
    @property
    def interviewee_list(self) -> dict[Times, list[Interviewee]]:
        return self._interviewee_list
    @property
    def priority(self):
        return len(self._time_slots)
    @property
    def appointment_type(self):
        return self._appointment_type
    def add_to_interviewee_list(self, interviewee: Interviewee, time : Times, max_alloc: int):
        if time not in self._interviewee_list.keys():
            self._interviewee_list[time] = []
        elif len(self._interviewee_list[time]) >= max_alloc:
            return 0
        interviewee.set_granted_slot(time)
        self._interviewee_list[time].append(interviewee)
        return 1    
    